# ARC Prize 2026 / ARC-AGI-3 — Autonomous Submission Agent
### Engineered & Powered by HBLLM Core Cognitive Architecture

This notebook deploys **MyAgent**, an autonomous inductive neuro-symbolic agent engineered with **HBLLM Core**.

- **Interactive / Commit Mode**: Evaluates MyAgent across the 25 benchmark games, reports live step-by-step level passes, and outputs the official tournament scorecard.
- **Competition Rerun Mode**: Connects to the competition gateway (`http://gateway:8001/`) via the official `ARC-AGI-3-Agents` harness to evaluate against hidden test environments.


In [ ]:
# ==============================================================================
# 1. INSTALL COMPETITION PACKAGES OFFLINE & CONFIGURE RUNTIME
# ==============================================================================
import os, sys, glob, subprocess

# Pin arc_agi ONLY_RESET_LEVELS so that game progress is preserved on death
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["MPLBACKEND"] = "agg"

wheel_files = glob.glob("/kaggle/input/**/*.whl", recursive=True)
if wheel_files:
    wheel_dirs = sorted(list(set(os.path.dirname(w) for w in wheel_files)))
    for wd in wheel_dirs:
        print(f"✓ Installing offline wheels from: {wd}")
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", f"--find-links={wd}", "arc-agi", "arcengine", "python-dotenv"], check=False)


In [ ]:
# ==============================================================================
# 2. LOCATE & MOUNT HBLLM CORE ENGINE
# ==============================================================================
import os, sys, shutil

package_found = False
hbllm_dir = None
for root, dirs, _ in os.walk("/kaggle/input"):
    if "kaggle_submission" in dirs and "hbllm" in dirs:
        hbllm_dir = root
        if root not in sys.path:
            sys.path.insert(0, root)
        print(f"✓ Found and mounted HBLLM Core from: {root}")
        sub_py = os.path.join(root, "kaggle_submission", "submission.py")
        if os.path.exists(sub_py):
            shutil.copy(sub_py, "/tmp/my_agent.py")
            print("✓ Exported agent template to /tmp/my_agent.py")
        package_found = True
        break

if not package_found:
    raise RuntimeError("Could not find HBLLM Core dataset! Make sure hbllm-arc-agi-3 is attached under Input.")


In [ ]:
# ==============================================================================
# 3. INTERACTIVE TOURNAMENT BENCHMARK EVALUATION (25 GAMES)
# ==============================================================================
import pandas as pd
from arc_agi import Arcade, OperationMode
from kaggle_submission.submission import MyAgent

scorecard = {}
tournament_rows = []

# Run the live benchmark tournament in interactive/commit mode
if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    env_dir = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"
    arcade = Arcade(operation_mode=OperationMode.OFFLINE, environments_dir=env_dir)
    agent = MyAgent()
    
    target_games = sorted(list(set(env_info.game_id.split("-")[0] for env_info in arcade.available_environments)))
    if not target_games:
        target_games = ["tr87", "su15", "cd82", "wa30"]
        
    print(f"🎮 Evaluating MyAgent (HBLLM Core) across {len(target_games)} games...\n")
    MAX_STEPS_PER_GAME = 120
    
    for idx, gid in enumerate(target_games):
        env = arcade.make(gid)
        if env is None:
            continue
        
        frame = env.reset()
        agent.reset_episode()
        history = []
        
        for s in range(MAX_STEPS_PER_GAME):
            action = agent.choose_action(history, frame)
            history.append(frame)
            
            # Extract data dictionary for complex actions (ACTION6)
            data = getattr(action, "data", None)
            if data is None and hasattr(action, "action_data") and action.action_data is not None:
                ad = action.action_data
                data = {"x": getattr(ad, "x", 0), "y": getattr(ad, "y", 0)}
            if data is None and hasattr(action, "is_complex") and action.is_complex():
                data = {"x": 0, "y": 0}
                
            frame = env.step(action, data=data)
            if frame is None:
                print(f"  [{gid:<4}] Step returned None, ending game.")
                break
                
            state_str = getattr(frame.state, "name", str(frame.state))
            if frame.levels_completed > getattr(agent, "current_levels_completed", 0):
                print(f"  [{gid:<4}] 🎉 Level {frame.levels_completed}/{frame.win_levels} passed at step {s+1}!")
                
            if state_str == "WIN" or frame.levels_completed >= frame.win_levels:
                print(f"  [{gid:<4}] 🏆 ALL LEVELS COMPLETED in {s+1} steps!")
                break
            elif state_str == "GAME_OVER":
                print(f"  [{gid:<4}] Game over at step {s+1}.")
                break
                
        if frame is not None:
            scorecard[gid] = f"{frame.levels_completed}/{frame.win_levels}"
            tournament_rows.append({
                "row_id": f"{idx+1}_0",
                "game_id": gid,
                "end_of_game": True,
                "score": frame.levels_completed
            })
        
    print("\n" + "="*60 + "\n🏁 FINAL TOURNAMENT SCORECARD:\n" + "="*60)
    for gid, score in scorecard.items():
        print(f"  {gid:<10}: {score}")


In [ ]:
# ==============================================================================
# 4. OFFICIAL COMPETITION RERUN (PHASE B: GATEWAY SIDECAR)
# ==============================================================================
import os, sys, shutil, subprocess

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    print("🚀 Kaggle Competition Rerun Detected! Initializing Gateway Connection...")
    os.environ["ONLY_RESET_LEVELS"] = "true"
    
    # 1. Wait for competition gateway sidecar
    print("⏳ Waiting for competition gateway sidecar at http://gateway:8001/api/games ...")
    subprocess.run(["curl", "--fail", "--retry", "999", "--retry-all-errors", "--retry-delay", "5", "--retry-max-time", "600", "http://gateway:8001/api/games"], check=True)
    print("✓ Gateway sidecar is alive and reachable!")

    # 2. Copy the official competition framework into working directory
    agents_dir = "/kaggle/working/ARC-AGI-3-Agents"
    if os.path.exists(agents_dir):
        shutil.rmtree(agents_dir)
    shutil.copytree("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents", agents_dir)
    print(f"✓ Copied ARC-AGI-3-Agents framework to {agents_dir}")

    # 3. Drop MyAgent into the framework
    target_agent_path = os.path.join(agents_dir, "agents", "templates", "my_agent.py")
    shutil.copy("/tmp/my_agent.py", target_agent_path)
    print(f"✓ Installed MyAgent into framework at {target_agent_path}")

    # 4. Register MyAgent in the framework agent registry
    init_py = os.path.join(agents_dir, "agents", "__init__.py")
    with open(init_py, "w") as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
""")
    print("✓ Registered MyAgent in agent registry")

    # 5. Configure environment for gateway sidecar connection with ONLY_RESET_LEVELS
    env_file = os.path.join(agents_dir, ".env")
    with open(env_file, "w") as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
ONLY_RESET_LEVELS=true
""")
    print("✓ Configured .env for gateway online mode with level preservation")

    # 6. Execute framework tournament against gateway
    print("🎮 Executing ARC-AGI-3 tournament against competition gateway...")
    env_vars = os.environ.copy()
    env_vars["MPLBACKEND"] = "agg"
    env_vars["ONLY_RESET_LEVELS"] = "true"
    if "hbllm_dir" in locals() and hbllm_dir:
        env_vars["PYTHONPATH"] = f"{hbllm_dir}:{env_vars.get("PYTHONPATH", "")}"
    result = subprocess.run([sys.executable, "main.py", "--agent", "myagent"], cwd=agents_dir, env=env_vars)
    print(f"✓ Competition tournament finished with exit code {result.returncode}")


In [ ]:
# ==============================================================================
# 5. EMIT / VERIFY SUBMISSION PARQUET FILE
# ==============================================================================
import os
import pandas as pd

if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    # In interactive/commit mode: emit formatted submission.parquet
    if tournament_rows:
        df_sub = pd.DataFrame(tournament_rows)
    else:
        df_sub = pd.DataFrame(
            data=[["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"]
        )
    df_sub.to_parquet("/kaggle/working/submission.parquet", index=False)
    print(f"✓ Generated /kaggle/working/submission.parquet with {len(df_sub)} game entries!")
    print(df_sub.head(10))
else:
    # In competition rerun mode: gateway emitted submission.parquet; verify it
    if os.path.exists("/kaggle/working/submission.parquet"):
        sub_df = pd.read_parquet("/kaggle/working/submission.parquet")
        print(f"✓ Official submission.parquet generated by gateway: {len(sub_df)} records!")
        print(sub_df.head(10))
    else:
        print("⚠️ Warning: Gateway submission.parquet not found in /kaggle/working/! Emitting compliant fallback...")
        df_sub = pd.DataFrame(
            data=[["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"]
        )
        df_sub.to_parquet("/kaggle/working/submission.parquet", index=False)
        print("✓ Emitted fallback /kaggle/working/submission.parquet to ensure valid submission")
